# 03 — Leaf Area Quantification & Disease Extent Estimation

Classical computer-vision analysis on the PlantVillage Kaggle dataset.

1. Leaf segmentation from colour images (HSV thresholding + morphology)
2. Leaf mask extraction from pre-segmented images
3. Leaf area measurement (pixel count + shape statistics)
4. Disease area estimation: healthy vs affected tissue
5. Qualitative comparison: healthy vs diseased leaves
6. Batch analysis across multiple samples

**Two complementary segmentation methods**:
- **Method A** — `color/` config: HSV thresholding → morphological cleanup → contour analysis
- **Method B** — `segmented/` config: threshold non-black pixels (background already removed)

In [ ]:
# Install dependencies and download dataset
!pip install -q datasets torch torchvision timm opencv-python matplotlib seaborn tqdm
!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --unzip -p /content/plantvillage

In [ ]:
import os, glob, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

DATA_COLOR = '/content/plantvillage/color'
DATA_SEG   = '/content/plantvillage/segmented'

print('Libraries loaded.')

## Helper functions (segmentation & visualisation)

In [ ]:
# ── Internal helpers ─────────────────────────────────────────────────────────
def _pil_to_bgr(image):
    return cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2BGR)


def _largest_component(binary_mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    if num_labels <= 1:
        return binary_mask
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(labels == largest, 255, 0).astype(np.uint8)


# ── Segmentation ─────────────────────────────────────────────────────────────
def segment_leaf_from_color(image):
    """HSV thresholding + morphology on a colour PIL image."""
    bgr = _pil_to_bgr(image)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([10,  25,  25])
    upper = np.array([90, 255, 255])
    mask  = cv2.inRange(hsv, lower, upper)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    return _largest_component(mask)


def segment_leaf_from_segmented(image):
    """Threshold non-black pixels from a pre-segmented PIL image."""
    arr  = np.array(image.convert('RGB'))
    mask = (arr.max(axis=2) > 20).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return _largest_component(mask)


# ── Area & shape ─────────────────────────────────────────────────────────────
def compute_leaf_area(mask):
    return int(np.count_nonzero(mask))


def get_contour_stats(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return {}
    c         = max(contours, key=cv2.contourArea)
    area      = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    circularity = (4.0 * np.pi * area / (perimeter ** 2)) if perimeter > 0 else 0.0
    return {
        'contour_area': float(area),
        'perimeter':    float(perimeter),
        'bounding_box': (int(x), int(y), int(w), int(h)),
        'circularity':  round(float(circularity), 4),
        'aspect_ratio': round(float(w) / float(h), 4) if h > 0 else 0.0,
    }


# ── Disease estimation ───────────────────────────────────────────────────────
def estimate_disease_area(color_image, segmented_image):
    """
    Estimate disease % using narrow-green HSV mask vs total leaf area.
    Returns dict: total_px, healthy_px, diseased_px, disease_pct.
    """
    total_mask = segment_leaf_from_segmented(segmented_image)
    total_px   = compute_leaf_area(total_mask)
    if total_px == 0:
        return {'total_px': 0, 'healthy_px': 0, 'diseased_px': 0, 'disease_pct': 0.0}

    bgr = _pil_to_bgr(color_image)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lower_healthy = np.array([30, 50, 40])
    upper_healthy = np.array([85, 255, 255])
    healthy_mask  = cv2.inRange(hsv, lower_healthy, upper_healthy)
    healthy_in_leaf = cv2.bitwise_and(healthy_mask, healthy_mask, mask=total_mask)
    healthy_px  = compute_leaf_area(healthy_in_leaf)
    diseased_px = max(0, total_px - healthy_px)
    disease_pct = round(100.0 * diseased_px / total_px, 2)
    return {
        'total_px':    total_px,
        'healthy_px':  healthy_px,
        'diseased_px': diseased_px,
        'disease_pct': disease_pct,
    }


# ── Visualisation ─────────────────────────────────────────────────────────────
def overlay_mask(image, mask, color=(0, 200, 0), alpha=0.40):
    """Blend binary mask over PIL image. Returns uint8 RGB numpy array."""
    img_arr  = np.array(image.convert('RGB')).copy()
    coloured = img_arr.copy()
    coloured[mask > 0] = color
    return cv2.addWeighted(img_arr, 1 - alpha, coloured, alpha, 0)


def plot_segmentation(color_img, seg_img, leaf_mask, disease_stats, title=''):
    """4-panel: original | pre-segmented | mask overlay | disease pie."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(color_img);             axes[0].set_title('Original Colour');  axes[0].axis('off')
    if seg_img is not None:
        axes[1].imshow(seg_img);           axes[1].set_title('Pre-Segmented')
    else:
        axes[1].text(0.5, 0.5, 'N/A', ha='center', va='center')
    axes[1].axis('off')

    total = disease_stats.get('total_px', 0)
    axes[2].imshow(overlay_mask(color_img, leaf_mask))
    axes[2].set_title(f'Leaf Mask\nArea: {total:,} px'); axes[2].axis('off')

    dpct = disease_stats.get('disease_pct', 0.0); hpct = 100 - dpct
    axes[3].pie([hpct, dpct],
                labels=[f'Healthy\n{hpct:.1f}%', f'Affected\n{dpct:.1f}%'],
                colors=['#4CAF50', '#F44336'], startangle=90)
    axes[3].set_title('Tissue Composition')

    if title:
        fig.suptitle(title, fontsize=11, fontweight='bold')
    plt.tight_layout()
    return fig

print('Helper functions defined.')

## 1. Build paired image list (color ↔ segmented)

In [ ]:
class_names = sorted(os.listdir(DATA_COLOR))

# Build a list of matched (color_path, seg_path, class_name, crop, disease) dicts
paired_samples = []
for cls in class_names:
    color_dir = os.path.join(DATA_COLOR, cls)
    seg_dir   = os.path.join(DATA_SEG,   cls)
    if not os.path.isdir(seg_dir):
        continue
    shared = sorted(set(os.listdir(color_dir)) & set(os.listdir(seg_dir)))
    parts  = cls.split('___')
    crop    = parts[0]
    disease = parts[1] if len(parts) > 1 else 'Unknown'
    for fname in shared:
        paired_samples.append({
            'color_path': os.path.join(color_dir, fname),
            'seg_path':   os.path.join(seg_dir,   fname),
            'class_name': cls,
            'crop':       crop,
            'disease':    disease,
        })

print(f'Paired samples (color+segmented): {len(paired_samples):,}')
print(f'Example entry: {paired_samples[5]}')

## 2. Method A — Segmentation from colour image (HSV)

Converts to HSV colour space and thresholds the `[10°–90°]` hue range to detect leaf tissue, then applies morphological operations.

In [ ]:
sample = paired_samples[5]
color_img = Image.open(sample['color_path']).convert('RGB')
label_str = sample['class_name']

mask_a = segment_leaf_from_color(color_img)
area_a = compute_leaf_area(mask_a)
stats  = get_contour_stats(mask_a)

print(f'Class        : {label_str}')
print(f'Leaf area    : {area_a:,} px')
print(f'Circularity  : {stats.get("circularity", "N/A")}')
print(f'Aspect ratio : {stats.get("aspect_ratio", "N/A")}')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(color_img);                       axes[0].set_title('Original colour'); axes[0].axis('off')
axes[1].imshow(mask_a, cmap='gray');             axes[1].set_title('Leaf mask (Method A)'); axes[1].axis('off')
axes[2].imshow(overlay_mask(color_img, mask_a)); axes[2].set_title(f'Mask overlay\nArea: {area_a:,} px'); axes[2].axis('off')

fig.suptitle(f'{label_str} — HSV Segmentation', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('method_a_segmentation.png', bbox_inches='tight')
plt.show()

## 3. Method B — Segmentation from pre-segmented image

In [ ]:
seg_img = Image.open(sample['seg_path']).convert('RGB')
mask_b  = segment_leaf_from_segmented(seg_img)
area_b  = compute_leaf_area(mask_b)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(seg_img);                            axes[0].set_title('Pre-segmented image'); axes[0].axis('off')
axes[1].imshow(mask_b, cmap='gray');                axes[1].set_title('Leaf mask (Method B)'); axes[1].axis('off')
axes[2].imshow(overlay_mask(seg_img, mask_b, color=(0, 120, 255), alpha=0.35))
axes[2].set_title(f'Mask overlay\nArea: {area_b:,} px'); axes[2].axis('off')

fig.suptitle(f'{label_str} — Pre-Segmented Image Mask', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('method_b_segmentation.png', bbox_inches='tight')
plt.show()

print(f'Method A area: {area_a:>10,} px')
print(f'Method B area: {area_b:>10,} px')
diff = abs(area_a - area_b)
print(f'Difference   : {diff:>10,} px  ({100*diff/max(area_a,area_b):.1f}%)')

## 4. Disease area estimation

In [ ]:
# Find a diseased sample (disease field is not 'healthy')
diseased_sample = next(
    s for s in paired_samples if s['disease'].lower() != 'healthy'
)

color_d = Image.open(diseased_sample['color_path']).convert('RGB')
seg_d   = Image.open(diseased_sample['seg_path']).convert('RGB')
label_d = diseased_sample['class_name']

disease_stats = estimate_disease_area(color_d, seg_d)

print(f'Class        : {label_d}')
print(f'Total leaf   : {disease_stats["total_px"]:>10,} px')
print(f'Healthy      : {disease_stats["healthy_px"]:>10,} px')
print(f'Diseased     : {disease_stats["diseased_px"]:>10,} px  ({disease_stats["disease_pct"]}%)')

fig = plot_segmentation(
    color_d, seg_d,
    segment_leaf_from_segmented(seg_d),
    disease_stats,
    title=f'{label_d} — Disease Area Analysis',
)
plt.savefig('disease_area.png', bbox_inches='tight')
plt.show()

## 5. Healthy vs Diseased — side-by-side comparison

In [ ]:
# Find a healthy leaf from the same crop
target_crop = diseased_sample['crop']
healthy_sample = next(
    (s for s in paired_samples
     if s['crop'] == target_crop and s['disease'].lower() == 'healthy'),
    paired_samples[0],
)

color_h = Image.open(healthy_sample['color_path']).convert('RGB')
seg_h   = Image.open(healthy_sample['seg_path']).convert('RGB')
label_h = healthy_sample['class_name']

stats_h = estimate_disease_area(color_h, seg_h)
stats_d = estimate_disease_area(color_d, seg_d)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for row, (c_img, s_img, stats, lbl) in enumerate([
    (color_h, seg_h, stats_h, label_h),
    (color_d, seg_d, stats_d, label_d),
]):
    mask = segment_leaf_from_segmented(s_img)
    axes[row][0].imshow(c_img)
    axes[row][0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row][0].set_ylabel('Healthy' if row == 0 else 'Diseased',
                             fontsize=11, fontweight='bold')
    axes[row][0].axis('off')

    axes[row][1].imshow(s_img)
    axes[row][1].set_title('Pre-segmented' if row == 0 else '', fontsize=9)
    axes[row][1].axis('off')

    area = compute_leaf_area(mask)
    axes[row][2].imshow(overlay_mask(c_img, mask))
    axes[row][2].set_title(f'Leaf mask\n{area:,} px', fontsize=9)
    axes[row][2].axis('off')

    dpct = stats['disease_pct']
    axes[row][3].pie(
        [100 - dpct, dpct],
        labels=[f'Healthy\n{100-dpct:.1f}%', f'Affected\n{dpct:.1f}%'],
        colors=['#4CAF50', '#F44336'], startangle=90,
    )
    axes[row][3].set_title('Tissue composition' if row == 0 else '', fontsize=9)

fig.suptitle(f'Healthy ({label_h}) vs Diseased ({label_d})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('healthy_vs_diseased.png', bbox_inches='tight')
plt.show()

## 6. Batch analysis — disease percentage distribution

In [ ]:
N_BATCH = 200
batch   = random.sample(paired_samples, min(N_BATCH, len(paired_samples)))
results = []

for s in batch:
    color_img = Image.open(s['color_path']).convert('RGB')
    seg_img   = Image.open(s['seg_path']).convert('RGB')
    stats  = estimate_disease_area(color_img, seg_img)
    is_healthy = s['disease'].lower() == 'healthy'
    results.append({
        'disease_pct': stats['disease_pct'],
        'is_healthy':  is_healthy,
        'leaf_area':   stats['total_px'],
    })

healthy_pcts  = [r['disease_pct'] for r in results if r['is_healthy']]
diseased_pcts = [r['disease_pct'] for r in results if not r['is_healthy']]

print(f'Analysed : {len(results)} samples')
print(f'Healthy  : {len(healthy_pcts):>3}  mean disease %: {np.mean(healthy_pcts):.1f}%')
print(f'Diseased : {len(diseased_pcts):>3}  mean disease %: {np.mean(diseased_pcts):.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(healthy_pcts,  bins=20, color='#4CAF50', alpha=0.8,
             edgecolor='white', label='Healthy')
axes[0].hist(diseased_pcts, bins=20, color='#F44336', alpha=0.8,
             edgecolor='white', label='Diseased')
axes[0].set_xlabel('Estimated disease area (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Disease Area Distribution', fontweight='bold')
axes[0].legend()

leaf_areas = [r['leaf_area'] for r in results]
axes[1].scatter(
    leaf_areas, [r['disease_pct'] for r in results],
    c=['#4CAF50' if r['is_healthy'] else '#F44336' for r in results],
    alpha=0.7, edgecolors='none',
)
axes[1].set_xlabel('Leaf area (px)')
axes[1].set_ylabel('Disease area (%)')
axes[1].set_title('Leaf Area vs Disease %', fontweight='bold')
green_patch = mpatches.Patch(color='#4CAF50', label='Healthy')
red_patch   = mpatches.Patch(color='#F44336', label='Diseased')
axes[1].legend(handles=[green_patch, red_patch])

plt.tight_layout()
plt.savefig('disease_distribution.png', bbox_inches='tight')
plt.show()

## 7. Shape statistics — healthy vs diseased leaves

In [ ]:
shape_records = []
for s, r in zip(batch, results):
    seg_img = Image.open(s['seg_path']).convert('RGB')
    mask    = segment_leaf_from_segmented(seg_img)
    cs      = get_contour_stats(mask)
    if cs:
        cs['is_healthy'] = r['is_healthy']
        shape_records.append(cs)

circ_h = [r['circularity']  for r in shape_records if r['is_healthy']]
circ_d = [r['circularity']  for r in shape_records if not r['is_healthy']]
ar_h   = [r['aspect_ratio'] for r in shape_records if r['is_healthy']]
ar_d   = [r['aspect_ratio'] for r in shape_records if not r['is_healthy']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([circ_h, circ_d], labels=['Healthy', 'Diseased'],
                patch_artist=True,
                boxprops=dict(facecolor='#A8D8A8'),
                medianprops=dict(color='black', linewidth=2))
axes[0].set_title('Circularity'); axes[0].set_ylabel('Circularity (4πA/P²)')

axes[1].boxplot([ar_h, ar_d], labels=['Healthy', 'Diseased'],
                patch_artist=True,
                boxprops=dict(facecolor='#F4A8A8'),
                medianprops=dict(color='black', linewidth=2))
axes[1].set_title('Aspect Ratio'); axes[1].set_ylabel('Aspect ratio (w/h)')

plt.suptitle('Leaf Shape Statistics: Healthy vs Diseased', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shape_stats.png', bbox_inches='tight')
plt.show()

print(f'Circularity — Healthy : {np.mean(circ_h):.3f} ± {np.std(circ_h):.3f}')
print(f'Circularity — Diseased: {np.mean(circ_d):.3f} ± {np.std(circ_d):.3f}')

---
## Summary

### Techniques demonstrated
| Technique | Implementation |
|-----------|---------------|
| HSV thresholding | `segment_leaf_from_color()` — broad hue range |
| Morphological ops | `cv2.morphologyEx` CLOSE + OPEN |
| Connected components | Retain only the largest region |
| Contour analysis | Area, perimeter, circularity, aspect ratio |
| Disease estimation | Narrow-green HSV mask vs full leaf mask |

### Key observations
- Healthy leaves show **lower disease %** than diseased ones (as expected).
- Single-colour threshold + morphology is highly effective for PlantVillage's controlled-background images.
- The pre-segmented config provides ground-truth-quality masks; the HSV method generalises better to field images.

### Possible improvements
- Fine-tune HSV thresholds per crop species
- Train a lightweight U-Net for precise lesion segmentation
- Calibrate pixel area to real-world cm² using a reference object